# Baseline Failure Analysis (1a)

**Purpose**: establish overall baseline performance and identify where the model fails, providing the evidence base for the project hypothesis.

**Model**: U-Net + ResNet-34 encoder with baseline configuration

**Input**: per-frame results CSV exported by the baseline training notebook, joined with the annotator metadata spreadsheet.

## Scope

This notebook should address the question: **Does the baseline model fail uniformly, or are its failures concentrated in identifiable conditions?**

It covers:
1. Data overview: is the held-out result set balanced enough to trust the breakdowns below?
2. IoU and failure rate by category: annotator, surgery type, video resolution, robotic vs. manual, size stratum, thinness stratum, and challenge label
3. Summary figures for each breakdown

## Out of scope (see Notebook 01b)

- Case-mix-adjusted annotator checks
- Interaction screens (e.g. size × challenge, annotator × size)
- The per-frame failure catalogue
- The directional effect screen

These are covered in more detail in **Notebook 01b**, once the weak categories have been identified here.

## 1. Install dependencies
Run this cell to install the packages required for this notebook, pinned in `requirements.txt`.

In [ ]:
from pathlib import Path

_req = Path("requirements.txt")
if not _req.exists():
	_req = Path("01_baseline_analysis") / "requirements.txt"
if not _req.exists():
	raise FileNotFoundError(f"requirements.txt not found from cwd={Path.cwd()}")
get_ipython().run_line_magic("pip", f"install -q -r {_req.as_posix()}")


## 2. Setup and configuration
Imports the shared helpers, then sets the file paths and output directory for this notebook.

In [ ]:
import sys
from pathlib import Path

_NB_DIR = Path.cwd()
if not (_NB_DIR / "baseline_common.py").exists():
	_NB_DIR = _NB_DIR / "01_baseline_analysis"
if not (_NB_DIR / "baseline_common.py").exists():
	raise ModuleNotFoundError(f"baseline_common.py not found from cwd={Path.cwd()}")
sys.path.insert(0, str(_NB_DIR.resolve()))

from typing import cast

import importlib
import matplotlib.pyplot as plt
import baseline_common
importlib.reload(baseline_common)  # pick up edits to baseline_common without a kernel restart
from baseline_common import *

# ── Data sources ──
# 00_model_training writes: gs://{CKPT_BUCKET}/experiments/{RUN_NAME}/{RUN_ID}/
CKPT_BUCKET = "training-checkpoints-REPLACE_K_NUMBER"
RUN_NAME    = "baseline-seed42-clean_cv_Unet_resnet34_bce_lr1e-04_p20"
RUN_ID      = "20260813-232105"
GCS_RUN_DIR = f"gs://{CKPT_BUCKET}/experiments/{RUN_NAME}/{RUN_ID}"
SHEET_ID    = "REPLACE_SHEET_ID"

CACHE_DIR  = Path("/tmp/laparoscopic_strata_cache")
OUTPUT_DIR = Path("/tmp/laparoscopic_strata_1a")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH     = fetch_per_image_results(GCS_RUN_DIR, CACHE_DIR)
ANNOTATIONS_PATH = fetch_annotations(SHEET_ID, CACHE_DIR)

print("GCS run:", GCS_RUN_DIR)
print("Results:", RESULTS_PATH)
print("Annotations:", ANNOTATIONS_PATH)
print("Output dir:", OUTPUT_DIR)


## 3. Load and pair data
Load the results CSV and annotation spreadsheet and join them on a frame key.

In [ ]:
frame_df, metadata_mismatches, invalid_annotations = load_and_pair(RESULTS_PATH, ANNOTATIONS_PATH)

display(metadata_mismatches)
display(frame_df[["key", "image", *META_COLS, "Status", "Notes"]].head())

## 4. Derived metadata and instrument size class
Compute the instrument size and thinness classes used throughout this notebook, plus the
failure flags used by every breakdown below. Size matches the training notebook's frame-level
area cuts (tiny < 4.5%, small < 8%, medium < 19%, large >= 19%). Thinness is the run's own
`thin_class`, cut at the 25th percentile of mean local width (`thin_cut_px`, 22.81 px at
512 x 512), so it is the same boundary every thin number in the dissertation is scored on.


In [ ]:
frame_df = add_derived_columns(frame_df)
frame_df, cutoffs, failure_thresholds = add_failure_flags(frame_df)

display(cutoffs)
display(frame_df["size_cls_label"].value_counts().sort_index().rename_axis("size_cls").reset_index(name="frames"))
display(frame_df["thin_cls_label"].value_counts().sort_index().rename_axis("thin_cls").reset_index(name="frames"))
display(cast(pd.Series, frame_df[FAILURE_FLAG_COLS + ["failure_any"]].mean()).rename("rate").reset_index().rename(columns={"index": "failure_flag"}))

## 5. Data overview
Check that the held-out result set is balanced enough to trust the breakdowns below.

In [ ]:
# == Row and video counts ==
overview = pd.DataFrame({
    "rows": [len(frame_df)],
    "videos": [frame_df["Video ID"].nunique()],
    "annotators": [frame_df["Annotator"].nunique()],
    "surgery_types": [frame_df["Surgery Type"].nunique()],
    "challenge_combos": [frame_df["challenge_combo"].nunique()],
    "no_tool_rows": [int(frame_df["no_tools_bool"].sum())],
    "robotic_rows": [int(frame_df["Robotic"].fillna(False).astype(bool).sum())],
})
display(overview)

# == Metric distributions ==
metric_summary = frame_df[METRICS + ["tool_area_pct", "fp_area_pct", "fp_to_tool_ratio", "pred_fp_share"]].describe(
    percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
).T
display(metric_summary[["count", "mean", "std", "min", "5%", "10%", "25%", "50%", "75%", "90%", "95%", "max"]])

# == Challenge combinations ==
print("Top challenge combinations")
display(frame_df["challenge_combo"].value_counts().head(20).rename_axis("challenge_combo").reset_index(name="frames"))

# == Excluded annotations ==
# Rows dropped at the join in section 4 (not used in any breakdown)
print("Invalid or deleted annotations not used in model results")
if len(invalid_annotations):
    display(invalid_annotations[["Video ID", "File Name", "Surgery Type", "Video Resolution", "Robotic", "Annotator", "Challenges", "No Tools", "Status", "Notes"]].head(30))
else:
    print("None")

## 6. Challenge label breakdown
IoU and failure rate for each individual challenge label (smoke, blood, motion blur, overexposure, etc.), split out from the raw multi-label Challenges field.

In [ ]:
challenge_long = frame_df.copy()
challenge_long["challenge_token"] = challenge_long["challenge_tokens"].map(lambda xs: xs if xs else ["None"])
challenge_long = challenge_long.explode("challenge_token")
challenge_long["challenge_token"] = challenge_long["challenge_token"].map(lambda x: clean_label(x, empty="None"))
challenge_long["challenge_token_top"] = collapse_rare(challenge_long["challenge_token"], top_n=20, min_count=15)

challenge_token_table = summarize_groups(challenge_long, "challenge_token_top", min_rows=20, min_videos=1)
display_and_save(challenge_token_table, "challenge_token_table", OUTPUT_DIR, n=30)

## 7. Category breakdowns
IoU and failure rate by annotator, surgery type, video resolution, robotic vs. manual, size stratum, thinness stratum, and challenge combination.

In [ ]:
frame_screen = frame_df.copy()
frame_screen["Surgery Type Top"] = collapse_rare(frame_screen["Surgery Type"], top_n=25, min_count=25)
frame_screen["Resolution Top"] = collapse_rare(frame_screen["Video Resolution"], top_n=15, min_count=10)
frame_screen["Challenge Combo Top"] = collapse_rare(frame_screen["challenge_combo"], top_n=20, min_count=15)

screen_specs = [
    ("annotator_frame", "Annotator", 1),
    ("surgery_frame", "Surgery Type Top", MIN_GROUP_FRAMES),
    ("resolution_frame", "Resolution Top", MIN_GROUP_FRAMES),
    ("robotic_frame", "Robotic", MIN_GROUP_FRAMES),
    ("no_tools_frame", "No Tools", 1),
    ("size_frame", "size_cls_label", MIN_GROUP_FRAMES),
    ("thin_frame", "thin_cls_label", MIN_GROUP_FRAMES),
    ("challenge_combo_frame", "Challenge Combo Top", MIN_GROUP_FRAMES),
]

screen_tables = {}
for screen_name, screen_col, screen_min_rows in screen_specs:
    screen_table = summarize_groups(frame_screen, screen_col, min_rows=screen_min_rows, min_videos=1)
    screen_tables[screen_name] = screen_table
    print(f"\n{screen_name}")
    display_and_save(screen_table, screen_name, OUTPUT_DIR, n=25)

## 8. Summary figures
Charts translating the category breakdowns into a clear visual summary of where the baseline model fails.

In [ ]:
from matplotlib.ticker import PercentFormatter

VISUAL_DIR = OUTPUT_DIR / "figures"
VISUAL_DIR.mkdir(parents=True, exist_ok=True)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

COLORS = {
    "risk": PRIMARY,
    "volume": SECONDARY,
    "good": ACCENT,
    "warning": ACCENT_WARM,
    "gray": RULE,
}


def save_and_show(fig, filename):
    path = VISUAL_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    print(f"Saved {path}")
    if "agg" not in plt.get_backend().lower():
        plt.show()
    plt.close(fig)
    return path


def clean_failure_label(value):
    return str(value).replace("failure_", "").replace("_", " ").title()


def shorten_label(value, max_len=36):
    text = clean_label(value)
    return text if len(text) <= max_len else text[: max_len - 1] + "..."


def sorted_size_table(table, label_col="size_cls_label"):
    # orders "0_none" / "1_tiny" ... and the "1_thin" / "2_wide" labels on the same rule
    if table.empty or label_col not in table.columns:
        return table.copy()
    data = table.copy()
    order = data[label_col].astype(str).str.extract(r"^(\d+)")[0]
    data["_size_order"] = pd.to_numeric(order, errors="coerce")
    return data.sort_values(["_size_order", label_col]).drop(columns="_size_order")


def set_percent_axis(ax, axis="x"):
    formatter = PercentFormatter(1.0)
    if axis == "x":
        ax.xaxis.set_major_formatter(formatter)
    else:
        ax.yaxis.set_major_formatter(formatter)


def plot_metric_distributions(df):
    metric_specs = [
        ("iou", "IoU", [(0.50, "severe IoU"), (failure_thresholds["weak_iou"], "p10 weak IoU")], False),
        ("dice", "Dice", [], False),
        ("precision", "Precision", [(failure_thresholds["precision"], "p10 low precision")], False),
        ("hd95", "HD95", [(failure_thresholds["boundary"], "p90 boundary")], True),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax, (metric_col, label, thresholds, clip_tail) in zip(axes.flat, metric_specs):
        values = cast(pd.Series, pd.to_numeric(df[metric_col], errors="coerce")).replace([np.inf, -np.inf], np.nan).dropna()
        if values.empty:
            ax.set_axis_off()
            ax.set_title(f"{label}: no data")
            continue
        plot_values = values.copy()
        xlabel = label
        if clip_tail:
            upper = values.quantile(0.98)
            plot_values = values.clip(upper=upper)
            xlabel = f"{label} (values above p98 clipped)"
        ax.hist(plot_values, bins=40, color=COLORS["risk"], alpha=0.85, edgecolor="white")
        for threshold, threshold_label in thresholds:
            if pd.notna(threshold) and np.isfinite(threshold) and threshold <= plot_values.max():
                ax.axvline(threshold, color=COLORS["risk"], linestyle="--", linewidth=1.8, label=threshold_label)
        ax.set_title(f"{label} distribution")
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Frames")
        ax.grid(axis="y", alpha=0.25)
        if thresholds:
            ax.legend(frameon=False, fontsize=8)
    return save_and_show(fig, "metric_distributions.png")


def plot_failure_flag_rates(df):
    flag_cols = FAILURE_FLAG_COLS + ["failure_any"]
    rates = df[flag_cols].mean().sort_values()
    labels = [clean_failure_label(flag_col) for flag_col in rates.index]
    fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(rates))))
    bars = ax.barh(labels, rates.values, color=COLORS["risk"], alpha=0.9)
    limit = min(1.0, max(0.05, float(rates.max()) * 1.25))
    ax.set_xlim(0, limit)
    set_percent_axis(ax, "x")
    ax.set_title("Failure flag rates")
    ax.set_xlabel("Frame rate")
    ax.grid(axis="x", alpha=0.25)
    for bar, value in zip(bars, rates.values):
        ax.text(value + limit * 0.015, bar.get_y() + bar.get_height() / 2, f"{value:.1%}", va="center", fontsize=9)
    return save_and_show(fig, "failure_flag_rates.png")


def plot_size_profile(size_table):
    if size_table.empty:
        print("No data for size profile")
        return None
    data = sorted_size_table(size_table)
    labels = data["size_cls_label"].astype(str).tolist()
    x = np.arange(len(data))
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), gridspec_kw={"width_ratios": [1, 1, 1.2]})

    axes[0].bar(x, data["n_frames"], color=COLORS["volume"], alpha=0.9)
    axes[0].set_title("Frame volume")
    axes[0].set_ylabel("Frames")
    for i, value in enumerate(data["n_frames"]):
        axes[0].text(i, value, f"{int(value):,}", ha="center", va="bottom", fontsize=8)

    axes[1].bar(x, data["failure_any_rate"], color=COLORS["risk"], alpha=0.9)
    axes[1].set_title("Failure rate")
    axes[1].set_ylim(0, min(1.0, max(0.05, float(data["failure_any_rate"].max()) * 1.25)))
    set_percent_axis(axes[1], "y")
    for i, value in enumerate(data["failure_any_rate"]):
        axes[1].text(i, value, f"{value:.0%}", ha="center", va="bottom", fontsize=8)

    if {"iou_mean_ci_low", "iou_mean_ci_high"}.issubset(data.columns):
        yerr = np.vstack([
            data["iou_mean"] - data["iou_mean_ci_low"],
            data["iou_mean_ci_high"] - data["iou_mean"],
        ])
        axes[2].errorbar(x, data["iou_mean"], yerr=yerr, marker="o", color=COLORS["good"], capsize=4, linewidth=2)
    else:
        axes[2].plot(x, data["iou_mean"], marker="o", color=COLORS["good"], linewidth=2)
    axes[2].set_title("Mean IoU with 95% CI")
    axes[2].set_ylabel("Mean IoU")
    axes[2].set_ylim(max(0, float(data["iou_mean"].min()) - 0.08), min(1, float(data["iou_mean"].max()) + 0.08))

    for ax in axes:
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=25, ha="right")
        ax.grid(axis="y", alpha=0.25)
    return save_and_show(fig, "size_strata_profile.png")


def plot_decile_trend(df, value_col, name, filename, n_bins=10):
    """Frame-level IoU/failure trend across fine-grained deciles of value_col.
    Continuous performance curve via equal-count bins; complementary to the fixed
    training cuts in size_cls and thin_cls."""
    positive = df[df["has_tool_pixels"]].copy()
    if positive.empty:
        print(f"No tool-present frames for the {name} trend")
        return None
    positive["decile"] = pd.qcut(positive[value_col].rank(method="first"), q=n_bins, labels=False) + 1

    grouped = positive.groupby("decile").agg(
        n_frames=("key", "size"),
        value_mean=(value_col, "mean"),
        value_max=(value_col, "max"),
        iou_mean=("iou", "mean"),
        iou_std=("iou", "std"),
        failure_any_rate=("failure_any", "mean"),
    ).reset_index()
    grouped = add_mean_ci(grouped, "iou_mean", "iou_std", "n_frames", "iou_mean")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    x = grouped["decile"]

    yerr = np.vstack([
        grouped["iou_mean"] - grouped["iou_mean_ci_low"],
        grouped["iou_mean_ci_high"] - grouped["iou_mean"],
    ])
    axes[0].errorbar(x, grouped["iou_mean"], yerr=yerr, marker="o", color=COLORS["good"], capsize=4, linewidth=2)
    axes[0].set_title(f"Mean IoU by {name} decile (frame-level)")
    axes[0].set_xlabel(f"{name.capitalize()} decile (1 = lowest 10% of tool-present frames)")
    axes[0].set_ylabel("Mean IoU")
    axes[0].grid(axis="y", alpha=0.25)

    axes[1].bar(x, grouped["failure_any_rate"], color=COLORS["risk"], alpha=0.9)
    axes[1].set_title(f"Failure rate by {name} decile")
    axes[1].set_xlabel(f"{name.capitalize()} decile (1 = lowest 10% of tool-present frames)")
    set_percent_axis(axes[1], "y")
    axes[1].grid(axis="y", alpha=0.25)

    for ax in axes:
        ax.set_xticks(x)

    return save_and_show(fig, filename)


def plot_failure_reason_mix(df, group_col, title, filename, min_frames=MIN_GROUP_FRAMES):
    counts = df.groupby(group_col, dropna=False).size()
    valid_groups = counts[counts >= min_frames].index
    if len(valid_groups) == 0:
        print(f"No groups pass min_frames for {title}")
        return None
    data = df[df[group_col].isin(valid_groups)].copy()
    matrix = data.groupby(group_col, dropna=False)[FAILURE_FLAG_COLS].mean()
    if group_col in ("size_cls_label", "thin_cls_label"):
        matrix = sorted_size_table(matrix.reset_index(), group_col).set_index(group_col)
    else:
        matrix = matrix.loc[matrix.sum(axis=1).sort_values(ascending=False).index]
    matrix = matrix.rename(columns=clean_failure_label)
    values = matrix.values.astype(float)
    vmax = max(0.05, float(np.nanmax(values)))
    n_rows, n_cols = values.shape
    data_ma = np.ma.masked_invalid(values)

    # Fix for heatmap grid lines
    with plt.rc_context({"axes.grid": False}):
        fig, ax = plt.subplots(figsize=(max(8, 0.9 * n_cols), max(4, 0.45 * n_rows)))
        im = ax.pcolormesh(
            np.arange(n_cols + 1),
            np.arange(n_rows + 1),
            data_ma,
            cmap=SEQUENTIAL_CMAP,
            vmin=0,
            vmax=vmax,
            edgecolors="white",
            linewidths=0.8,
        )
        ax.invert_yaxis()
        ax.set_xticks(np.arange(n_cols) + 0.5)
        ax.set_xticklabels(matrix.columns.astype(str), rotation=35, ha="right")
        ax.set_yticks(np.arange(n_rows) + 0.5)
        ax.set_yticklabels(matrix.index.astype(str))
        ax.tick_params(which="both", length=0)
        ax.grid(visible=False, which="both", axis="both")
        ax.set_title(title)
        for row_i in range(n_rows):
            for col_i in range(n_cols):
                value = values[row_i, col_i]
                if np.isfinite(value):
                    text_color = "white" if value > vmax * 0.55 else RULE
                    ax.text(
                        col_i + 0.5,
                        row_i + 0.5,
                        f"{value:.0%}",
                        ha="center",
                        va="center",
                        color=text_color,
                        fontsize=8,
                    )
        cbar = fig.colorbar(im, ax=ax)
        cbar.ax.yaxis.set_major_formatter(PercentFormatter(1.0))
        cbar.ax.grid(False)
        cbar.set_label("Frame rate")
    return save_and_show(fig, filename)


def plot_priority_bars(table, label_col, title, filename, n=12):
    required = {label_col, "n_frames", "failure_any_rate"}
    if table.empty or not required.issubset(table.columns):
        print(f"No data for {title}")
        return None
    data = table.copy()
    data["failure_burden"] = data["failure_any_rate"] * data["n_frames"]
    data = data.replace([np.inf, -np.inf], np.nan).dropna(subset=["failure_burden"])
    if data.empty:
        print(f"No finite burden values for {title}")
        return None
    data = data.sort_values("failure_burden", ascending=False).head(n).sort_values("failure_burden")
    labels = [shorten_label(value, 42) for value in data[label_col]]
    fig, ax = plt.subplots(figsize=(11, max(4, 0.43 * len(data))))
    bars = ax.barh(labels, data["failure_burden"], color=COLORS["warning"], alpha=0.92)
    xmax = max(float(data["failure_burden"].max()) * 1.28, 1.0)
    ax.set_xlim(0, xmax)
    ax.set_title(title)
    ax.set_xlabel("Estimated failing frames (failure rate x frame count)")
    ax.grid(axis="x", alpha=0.25)
    for bar, (_, row) in zip(bars, data.iterrows()):
        label = f"{row['failure_any_rate']:.0%}, n={int(row['n_frames']):,}"
        ax.text(bar.get_width() + xmax * 0.015, bar.get_y() + bar.get_height() / 2, label, va="center", fontsize=8)
    return save_and_show(fig, filename)


visual_outputs = [
    plot_metric_distributions(frame_df),
    plot_failure_flag_rates(frame_df),
    plot_size_profile(screen_tables.get("size_frame", pd.DataFrame())),
    plot_decile_trend(frame_df, "tool_area_pct", "tool size", "size_tail_trend.png"),
    plot_decile_trend(frame_df, "width_px", "tool width", "width_tail_trend.png"),
    plot_failure_reason_mix(frame_df, "size_cls_label", "Failure reason mix by size stratum", "failure_reason_by_size.png"),
    plot_failure_reason_mix(frame_df, "thin_cls_label", "Failure reason mix by thinness stratum", "failure_reason_by_thinness.png"),
]

priority_specs = [
    ("annotator_frame", "Annotator", "Largest failure burden by annotator", "failure_burden_by_annotator.png"),
    ("surgery_frame", "Surgery Type Top", "Largest failure burden by surgery type", "failure_burden_by_surgery.png"),
    ("challenge_combo_frame", "Challenge Combo Top", "Largest failure burden by challenge combination", "failure_burden_by_challenge_combo.png"),
]
for spec_table_key, spec_label_col, spec_title, spec_filename in priority_specs:
    visual_outputs.append(plot_priority_bars(screen_tables.get(spec_table_key, pd.DataFrame()), spec_label_col, spec_title, spec_filename))

visual_outputs = [Path(path) for path in visual_outputs if path is not None]
if visual_outputs:
    display(pd.DataFrame({"figure": [path.name for path in visual_outputs], "path": [str(path) for path in visual_outputs]}))